# 5.1 Retrieve, inspect, and challenge

**Learning objectives:** normalize vectors; rank cosine similarity; evaluate retrieval manually; run an optional RemoteCLIP pipeline.

**How to learn:** predict - run - explain - change one thing. Spend 45–90 minutes here.

**Data:** the default worked example is deterministic synthetic practice data near Colorado. It is not SNOTEL, a watershed survey, a burn map, or a foundation-model output.


## Before you run

Image and text encoders map inputs into a shared vector space. Cosine similarity ranks candidates; it is not a calibrated probability. The default example uses transparent toy vectors so the ranking arithmetic is visible. Real RemoteCLIP inference is a separate, explicit lab.


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/"course.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course import OUT, points


## Work through retrieval by hand


In [2]:
names=np.array(['toy forest','toy snow','toy river'])
image_vectors=np.array([[1.,0.,.1],[0.,1.,.1],[.1,0.,1.]])
query=np.array([1.,0.,0.])
image_vectors/=np.linalg.norm(image_vectors,axis=1,keepdims=True)
query/=np.linalg.norm(query)
scores=image_vectors@query
ranking=np.argsort(-scores,kind='stable')
print(pd.DataFrame({'tile':names[ranking],'cosine_similarity':scores[ranking]}))
assert names[ranking[0]]=='toy forest'


         tile  cosine_similarity
0  toy forest           0.995037
1   toy river           0.099504
2    toy snow           0.000000


## Your turn

Try query `[0,1,0]`, then `[-1,-1,-1]`. Why does the system still return a best match when all candidates are poor? Design a relevance rubric for three real queries.

Use a new cell below to try this before opening the solution.


In [3]:
# Write your experiment here. The worked example above is already complete.


<details><summary>Solution and reasoning - open after trying</summary>

The first query retrieves toy snow. Ranking always has a maximum even when nothing is relevant. Define relevance before viewing results, judge top-k images, and report precision@k plus an absent-content query. Do not invent a universal cosine threshold.

</details>


## Check yourself

Expected: toy forest ranks first for `[1,0,0]`. Why cannot annual AlphaEarth vectors be compared directly to RemoteCLIP text vectors? They were not trained in the same shared space.

**Exit ticket:** write one result, one explanation, and one limitation. Save the notebook with your own observations.


## Transfer to real data

Complete `docs/CAPSTONE.md` using public RGB tiles with bounds, dates and provenance. Download the matching checkpoint using the command in `docs/REAL_DATA.md`. The code below batches images, normalizes both modalities and retains filenames. Inspect results visually; retrieval cannot establish water quality or damage causation.


## Optional real RemoteCLIP retrieval

Requires `requirements-models.txt`, `data/real/tiles/*.png` and `data/models/RemoteCLIP-ViT-B-32.pt`. Set the flag only after preparing those inputs.


In [4]:
RUN_REMOTECLIP=False
if RUN_REMOTECLIP:
    import torch, open_clip
    from PIL import Image
    checkpoint=ROOT/'data/models/RemoteCLIP-ViT-B-32.pt'
    tiles=sorted((ROOT/'data/real/tiles').glob('*.png'))
    if not checkpoint.exists() or not tiles:
        raise FileNotFoundError('Prepare the checkpoint and RGB PNG tiles using docs/REAL_DATA.md.')
    device='cuda' if torch.cuda.is_available() else 'cpu'
    model,_,preprocess=open_clip.create_model_and_transforms('ViT-B-32')
    model.load_state_dict(torch.load(checkpoint,map_location='cpu',weights_only=True),strict=True)
    model=model.to(device).eval()
    tokenizer=open_clip.get_tokenizer('ViT-B-32')
    queries=['burned forest','snow-covered ridge','river with visible sediment','an airport runway']
    vectors=[]
    with torch.inference_mode():
        for start in range(0,len(tiles),8):
            tensors=[]
            for p in tiles[start:start+8]:
                with Image.open(p) as im: tensors.append(preprocess(im.convert('RGB')))
            encoded=model.encode_image(torch.stack(tensors).to(device))
            vectors.append(torch.nn.functional.normalize(encoded,dim=-1).cpu())
        images=torch.cat(vectors)
        texts=torch.nn.functional.normalize(model.encode_text(tokenizer(queries).to(device)),dim=-1).cpu()
    similarity=texts@images.T
    rows=[]
    for q,score in zip(queries,similarity):
        for rank,index in enumerate(score.argsort(descending=True)[:min(5,len(tiles))],1):
            rows.append(dict(query=q,rank=rank,tile=str(tiles[int(index)].relative_to(ROOT)),similarity=float(score[index])))
    results=pd.DataFrame(rows)
    results.to_csv(OUT/'remoteclip_rankings.csv',index=False)
    print(results)
    fig,axes=plt.subplots(1,min(5,len(tiles)),figsize=(15,3),squeeze=False)
    for ax,index in zip(axes[0],similarity[0].argsort(descending=True)[:5]):
        with Image.open(tiles[int(index)]) as im: ax.imshow(im.convert('RGB'))
        ax.set_title(tiles[int(index)].stem);ax.axis('off')
    plt.show()
else:
    print('Toy retrieval complete. Real RemoteCLIP inference has not been run.')


Toy retrieval complete. Real RemoteCLIP inference has not been run.
